# Projeto: Detecção de Covid-19 via Padrões de Batimentos Cardíacos com KNN
Este notebook processa dados fisiológicos de 16 pacientes, extrai características estatísticas das séries temporais de batimentos cardíacos e aplica o algoritmo **KNN (K-Nearest Neighbors)** para classificar se o paciente testou positivo ou negativo para Covid-19, gerando uma visualização gráfica da fronteira de decisão.

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configuração de estilo dos gráficos
plt.style.use('ggplot')

In [11]:
# Defina a pasta onde estão guardados os seus 20 arquivos CSV e 20 arquivos JSON
# Se estiverem na mesma pasta do notebook, mantenha "./"
PASTA_DADOS = "dados"

# MAPEIE AQUI OS SEUS 20 PACIENTES REAIS (Substitua pelos nomes exatos dos seus arquivos)
mapeamento_pacientes = [
    {"json": "P1.json", "csv": "P1.csv"},
    {"json": "P2.json", "csv": "P2.csv"},
    {"json": "P3.json", "csv": "P3.csv"},
    {"json": "P4.json", "csv": "P4.csv"},
    {"json": "P5.json", "csv": "P5.csv"},
    {"json": "P6.json", "csv": "P6.csv"},
    {"json": "P7.json", "csv": "P7.csv"},
    {"json": "P8.json", "csv": "P8.csv"},
    {"json": "P9.json", "csv": "P9.csv"},
    {"json": "P10.json", "csv": "P10.csv"},
    {"json": "P11.json", "csv": "P11.csv"},
    {"json": "P12.json", "csv": "P12.csv"},
    {"json": "P13.json", "csv": "P13.csv"},
    {"json": "P14.json", "csv": "P14.csv"},
    {"json": "P15.json", "csv": "P15.csv"},
    {"json": "P16.json", "csv": "P16.csv"},
]

print(f"Configurados {len(mapeamento_pacientes)} pares de arquivos para processamento.")

Configurados 16 pares de arquivos para processamento.


In [12]:
dados_consolidados = []

for par in mapeamento_pacientes:
    nome_json = par["json"]
    nome_csv = par["csv"]
    
    caminho_json = os.path.join(PASTA_DADOS, nome_json)
    caminho_csv = os.path.join(PASTA_DADOS, nome_csv)
    
    # Pula o par caso algum dos arquivos esteja faltando na pasta
    if not os.path.exists(caminho_json) or not os.path.exists(caminho_csv):
        print(f"Aviso: Arquivo {nome_json} ou {nome_csv} não encontrado. Pulando...")
        continue
        
    # 1. LEITURA E REGRA DO DESFECHO (JSON)
    with open(caminho_json, 'r') as f:
        conteudo_json = json.load(f)
    
    # Extrai todas as respostas de alerta do histórico diário do paciente
    alertas = [item['val'] for item in conteudo_json['nightsignal']]
    
    # REGRA: Situação alarmante (1) se houver pelo menos um alerta '2'. Caso contrário, (0).
    situacao_alarmante = 1 if '2' in alertas else 0
    
    # 2. LEITURA E EXTRAÇÃO DE CARACTERÍSTICAS (CSV)
    df_csv = pd.read_csv(caminho_csv)
    media_hr = df_csv['HR_Value'].mean()
    std_hr = df_csv['HR_Value'].std() # Variabilidade dos batimentos cardíacos
    
    dados_consolidados.append({
        'Identificador': nome_json.split('.')[0],
        'Media_HR': media_hr,
        'Variabilidade_HR': std_hr,
        'Situacao_Alarmante': situacao_alarmante
    })

# Criação do DataFrame unificado do projeto
df_projeto = pd.DataFrame(dados_consolidados)
print(f"Sucesso! Base consolidada criada para {len(df_projeto)} pacientes.")
df_projeto

Aviso: Arquivo P1.json ou P1.csv não encontrado. Pulando...
Aviso: Arquivo P2.json ou P2.csv não encontrado. Pulando...
Aviso: Arquivo P3.json ou P3.csv não encontrado. Pulando...
Aviso: Arquivo P4.json ou P4.csv não encontrado. Pulando...
Aviso: Arquivo P5.json ou P5.csv não encontrado. Pulando...
Aviso: Arquivo P6.json ou P6.csv não encontrado. Pulando...
Aviso: Arquivo P7.json ou P7.csv não encontrado. Pulando...
Aviso: Arquivo P8.json ou P8.csv não encontrado. Pulando...
Aviso: Arquivo P9.json ou P9.csv não encontrado. Pulando...
Aviso: Arquivo P10.json ou P10.csv não encontrado. Pulando...
Aviso: Arquivo P11.json ou P11.csv não encontrado. Pulando...
Aviso: Arquivo P12.json ou P12.csv não encontrado. Pulando...
Aviso: Arquivo P13.json ou P13.csv não encontrado. Pulando...
Aviso: Arquivo P14.json ou P14.csv não encontrado. Pulando...
Aviso: Arquivo P15.json ou P15.csv não encontrado. Pulando...
Aviso: Arquivo P16.json ou P16.csv não encontrado. Pulando...
Sucesso! Base consolidada 

""
